<div style="background-color: #5397cbff ; padding: 20px; border-radius: 10px;">

<h1 style="color: black; text-align: center;">
    ML Project :  F1 lap time prediction
</h1>

<h3 style="color: black; text-align: center;">
    Model Implementation: XGBoost
</h3>

<h5 style="color: black; text-align: center;">
    Elisabeth Cognet, Nour El Khalili, Chatron Emilie -- MMN2
</h5>

</div>



# Ojective
This notebook develops and evaluates an XGBoost regression model to predict the fastest lap time in Formula 1 races.  
The objective is to test whether gradient boosting methods can outperform our existing Random Forest model by capturing non linear relationships between circuit features, driver characteristics, and car performance.  

The target variable for prediction is fastestLapTime, using the preprocessed dataset validated during the EDA stage.


In [7]:
!pip install xgboost


^C


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


In [24]:
X_train = pd.read_csv("data/processed/X_train_all.csv")
X_test = pd.read_csv("data/processed/X_test_all.csv")
y_train = pd.read_csv("data/processed/y_train_all.csv")
y_test = pd.read_csv("data/processed/y_test_all.csv")

# Convert y to 1D vectors if needed
y_train = y_train.squeeze()
y_test = y_test.squeeze()

X_train.head(), y_train.head()

(   grid  year     circuitRef     location  trackLength  trackTurns  \
 0     7  2020       istanbul     Istanbul        5.338          14   
 1    13  2016            spa          Spa        7.004          20   
 2     5  2005    silverstone  Silverstone        5.891          18   
 3    17  2021  red_bull_ring    Spielberg        4.318          10   
 4    23  2011         monaco  Monte-Carlo        3.337          19   
 
        driverRef driverNationality constructorRef constructorNationality  
 0           ocon            French        renault                 French  
 1  jolyon_palmer           British        renault                 French  
 2    barrichello         Brazilian        ferrari                Italian  
 3           ocon            French         alpine                 French  
 4         liuzzi           Italian            hrt                Spanish  ,
 0    101.006
 1    113.251
 2     82.302
 3     69.576
 4     81.566
 Name: fastestLapTime, dtype: float64)

In [25]:
xgb_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)

In [26]:
X_train.dtypes


grid                        int64
year                        int64
circuitRef                 object
location                   object
trackLength               float64
trackTurns                  int64
driverRef                  object
driverNationality          object
constructorRef             object
constructorNationality     object
dtype: object

In [27]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]
cat_cols


['circuitRef',
 'location',
 'driverRef',
 'driverNationality',
 'constructorRef',
 'constructorNationality']

In [35]:
y_train.name = "fastestLapTime"
y_test.name = "fastestLapTime"
y_train.name


'fastestLapTime'

In [38]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]

X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

target_col = y_train.name  # 'fastestLapTime'

# Combine X_train + y_train so grouping works
train_temp = X_train.copy()
train_temp[target_col] = y_train.values

for col in cat_cols:
    # Compute mean lap time per category
    target_means = train_temp.groupby(col)[target_col].mean()
    
    # Map training
    X_train_enc[col] = X_train[col].map(target_means)
    
    # Map test (unseen categories → global mean)
    X_test_enc[col] = X_test[col].map(target_means)
    X_test_enc[col].fillna(y_train.mean(), inplace=True)


In [43]:
xgb_model.fit(X_train_enc, y_train)
y_pred = xgb_model.predict(X_test_enc)


In [44]:
print(f"MAE = {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R^2 = {r2_score(y_test, y_pred):.4f}")


MAE = 2.4784
RMSE = 8.3269
R^2 = 0.6895


We will try to add two new features to see if we can boost our scores. They will be turns per km and average speed.

In [45]:
# Feature engineering
X_train["turns_per_km"] = X_train["trackTurns"] / X_train["trackLength"]
X_test["turns_per_km"] = X_test["trackTurns"] / X_test["trackLength"]

# avgSpeed_hint only for training signal (XGBoost learns relationships)
X_train["avgSpeed_hint"] = X_train["trackLength"] / y_train
X_test["avgSpeed_hint"] = X_test["trackLength"] / y_test



In [46]:
cat_cols = [col for col in X_train.columns if X_train[col].dtype == "object"]

X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

target_col = y_train.name  # 'fastestLapTime'

# Combine X_train + y_train so grouping works
train_temp = X_train.copy()
train_temp[target_col] = y_train.values

for col in cat_cols:
    # Compute mean lap time per category
    target_means = train_temp.groupby(col)[target_col].mean()
    
    # Map training
    X_train_enc[col] = X_train[col].map(target_means)
    
    # Map test (unseen categories → global mean)
    X_test_enc[col] = X_test[col].map(target_means)
    X_test_enc[col].fillna(y_train.mean(), inplace=True)


In [47]:
xgb_model.fit(X_train_enc, y_train)


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [48]:
y_pred = xgb_model.predict(X_test_enc)

print(f"MAE = {mean_absolute_error(y_test, y_pred):.4f}")
print(f"RMSE = {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")
print(f"R^2 = {r2_score(y_test, y_pred):.4f}")


MAE = 0.4529
RMSE = 3.4328
R^2 = 0.9472
